In [ ]:
using LinearAlgebra, MPSDynamics, Statistics, FFTW, StaticArrays
include("/Users/travail/Desktop/Stage_Quantic/Code_Julia/coupled_harmonic_oscillators_solver.jl")
include("/Users/travail/Desktop/Stage_Quantic/Code_Julia/Plot_setting.jl")

We rely on the article "Thorbeck, Ted, Alexander McDonald, O. Lanes, John Blair, George Keefe, Adam A. Stabile, Baptiste Royer, Luke C. G. Govia, et Alexandre Blais. « High-Fidelity Gates in a Transmon Using Bath Engineering for Passive Leakage Reset ». arXiv, 6 novembre 2024. https://doi.org/10.48550/arXiv.2411.04101." The bath is colored via the purcell filter, creating a pass-band filter. The frequency absorption for the bath is between $\omega_{min}$ and $\omega_{max}$. We will take $\Re(Z)=Z_0$ in the window and 0 outside, with a spooth step. The real part is non zero on a 2 GHz range.

For a Caldeira-Leggett approach: $\omega_m=\omega_{min}+m\frac{\omega_{max}-\omega_{min}}{n}$, $C_m=\frac{1}{\Re(Z)(\omega_m)\omega_m}$.

For the oscillator, we set the frequency in the middle of the window $\omega_0=3.5$ GHz and we choose the same capacitance order of magnitude as the one given for the mode $b$ in Lescanne, Raphaël, Marius Villiers, Théau Peronnin, Alain Sarlette, Matthieu Delbecq, Benjamin Huard, Takis Kontos, Mazyar Mirrahimi, et Zaki Leghtas. « Exponential Suppression of Bit-Flips in a Qubit Encoded in an Oscillator ». Nature Physics 16, nᵒ 5 (mai 2020): 509‑13. https://doi.org/10.1038/s41567-020-0824-x, so $E_C=73.5$ MHz and

\begin{equation}
    1/C=2E_C/e^2=2\times2\pi\frac{E_C}{h}\frac{\hbar}{e^2}=0.147\times2\pi GHz,
\end{equation}
where we set $\hbar=1$ and $e$=1.


It will be easier to work with $1/C$ and $1/C_m$.

Then, following Jeffrey, Evan, Daniel Sank, J. Y. Mutus, T. C. White, J. Kelly, R. Barends, Y. Chen, et al. « Fast Accurate State Measurement with Superconducting Qubits ». Physical Review Letters 112, nᵒ 19 (15 mai 2014): 190504. https://doi.org/10.1103/PhysRevLett.112.190504, we set the dissipation rate of the bath to $\kappa=1/20$ ns and following the lecture notes of Michel Devoret,
\begin{equation}
    Z_0=\frac{1}{2\kappa C}.
\end{equation}

The eigenfrequencie of the buffer mode is $\omega_b/2\pi\approx 5$ GHz, so that we take $\omega_{min}/2\pi=4$ GHz and $\omega_{max}/2\pi=6$ GHz.

We start by working at zero temperature

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞

#### Caracterization of the circuit #####
ω_sys=5*2*pi  # GHz
C_inv = 0.147*2*pi #ħ/e^2 GHz from Lescanne et al.
L = C_inv / ω_sys^2 # L in 1/GHz
Z_sys = sqrt(L*C_inv) # Characteristic impedance (dimensionless)
κ=1/20; # 1/ns from Jeffrey et al.

In [ ]:
function smoothstep(x)
    return x^3 * (x * (x * 6 - 15) + 10)
end

In [ ]:
function FFT_perso(ϕ,t)
    F = fftshift(fft(ϕ))
    fs = length(t)/(maximum(t)-minimum(t)) # sampling frequency
    freqs = fftshift(fftfreq(length(t), fs))
    return F, freqs
end

function comparison_simu(simu_list,E, E_params)
    ϕ_list = []
    q_list = []
    t_list = []
    E_list = []
    spectrum_list = []

    for simu in simu_list
        ϕ, q, t = simu
        push!(ϕ_list, ϕ)
        push!(q_list, q)
        push!(t_list, t)
        push!(E_list, E(ϕ, q, E_params))
        spectrum = FFT_perso(ϕ, t)
        push!(spectrum_list, spectrum)
    end

    return(E_list, spectrum_list)
end

function Energy_linear(ϕ, q, (L, C_inv))
    """
    INPUT :
    ϕ : vector of the field ϕ
    q : vector of the field q
    L : inductance
    C_inv : inverse capacitance in GHz

    OUTPUT :
    Energy of the linear system
    """
    return q.^2 /2 *C_inv + ϕ.^2 / (2*L)
end;

In [ ]:
function Ohmic_bath(;bath_params=(ωc_dwn=0.0*2*pi, ωc_up=10*2*pi), κ=1/20, C_inv=0.147*2*pi, plot_bool= false)
    """
    INPUT :
    bath_params 
        ωc_dwn : lower cut-off frequency of the spectral density J(ω) in GHz
        ωc_up : upper cut-off frequency of the spectral density J(ω) in GHz

    OUTPUT :
    AB : Array of the segments of the frequency ranges where the spectral density J(ω) is defined.
         Each segment is an array with two elements [ω_min ω_max] in GHz.
    Jω_fct : spectral density J(ω)
    """

    ωc_dwn, ωc_up = bath_params.ωc_dwn, bath_params.ωc_up

    ### Caracterization of the impedance #####
    α = 2*κ/C_inv # no unit

    # We design the impedance to be a smooth pass-band filter
    ω_max = ωc_up-.2*2*pi # Lower cut-off of the spectral density J(ω), GHz
    ω_min = ωc_dwn+.2*2*pi # Cut-off of the spectral density J(ω), GHz


    # AB segments correspond to the different frequency ranges where the spectral density is defined.
    # It corresponds to i=1 ; i=2 and i=3 of the Jω_fct(ω,i) function.

    AB = [[-Inf ωc_dwn];[ωc_dwn ω_min];[ω_min ω_max];[ω_max ωc_up];[ωc_up Inf]]
    #AB=[[-Inf ω_min];[ω_min ω_max];[ω_max Inf]]


    function Jω_fct(ω,i)
        if i==1 # First segment of AB (here [-Inf ω_min]), zero for negative frequencies when β == Inf
            y = 0
        elseif i==2 # Second segment of AB (here [ω_min ω_max])
            y = @. smoothstep((ω - ωc_dwn) / (ω_min - ωc_dwn))
        elseif i==3 # Third segment of AB (here [ω_min ω_max])
            y = 1
        elseif i==4 # Fourth segment of AB (here [ω_max ωc_up])``
            y = @. (1 - smoothstep((ω - ω_max) / (ωc_up - ω_max)))
        elseif i==5 # Fifth segment of AB (here [ωc_up Inf])
            y = 0
        end
        return y*α.*ω
    end
    if plot_bool
        ω = range(ωc_dwn, stop=ω_max, length=1000)
        J_values = [Jω_fct(ω[i], 3)*ω[i] for i in 1:length(ω)]
        plot(ω, J_values, label="J(ω)", xlabel="ω (GHz)", ylabel="J(ω)")
    end
    return AB, Jω_fct
end

function Exp_decay(;bath_params=(ω_c=50*2pi, ω_cut_off=90), κ=1/20, C_inv=0.147*2*pi, plot_bool= false)
    """
    INPUT :
    bath_params 
        ω_c : characteristic frequency of the exponential decay in GHz
        ω_cut_off : cut-off frequency of the spectral density J(ω) in GHz

    OUTPUT :
    AB : Array of the segments of the frequency ranges where the spectral density J(ω) is defined.
         Each segment is an array with two elements [ω_min ω_max] in GHz.
    Jω_fct : spectral density J(ω)
    """

    ω_c, ω_cut_off = bath_params.ω_c, bath_params.ω_cut_off
    
    ### Caracterization of the impedance #####
    α = 2*κ/C_inv # no unit

    AB=[[-Inf 0];[0 ω_cut_off];[ω_cut_off Inf]]

    function J_exp_decay(ω,i)
        if i==1 # First segment of AB (here [-Inf 0]), zero for negative frequencies when β == Inf
            y = 0
        elseif i==2 # Second segment of AB (here [0 ω_cut_off])
            y = @.α*ω*exp(-ω/ω_c)
        elseif i==3 # Third segment of AB (here [ω_cut_off Inf])
            y=0
        end
        return y
    end
    if plot_bool
        ω = range(0, stop=ω_cut_off, length=1000)
        J_values = [J_exp_decay(ω[i], 2) for i in 1:length(ω)]
        plot(ω, J_values, label="J(ω)", xlabel="ω (GHz)", ylabel="J(ω)")
    end
    return AB, J_exp_decay
end

function Drude_lorentz(;bath_params=(Ω=50*2pi, ω_cut_off=100), κ=1/20, C_inv=0.147*2*pi, plot_bool= false)
    """
    INPUT :
    bath_params
        Ω : characteristic frequency of the Drude-Lorentz function in GHz
        ω_cut_off : cut-off frequency of the spectral density J(ω) in GHz

    OUTPUT :
    onsite_energies : vector of on-site energies ϵ_i of the chain
    hopping_energies : vector of hopping energies t_i of the chain
    system_chain_coupling : coupling between the system and the chain
    """

    Ω, ω_cut_off = bath_params.Ω, bath_params.ω_cut_off

    AB_drude= [[-Inf 0];[0 ω_cut_off];[ω_cut_off Inf]]
    function J_drude(ω,i)
        if i==1 # First segment of AB (here [-Inf 0]), zero for negative frequencies when β == Inf
            y = 0
        elseif i==2 # Second segment of AB (here [0 ω_cut_off])
            y = @. 2*κ/(C_inv)*ω*Ω^2/((ω^2+Ω^2))
        elseif i==3 # Third segment of AB (here [ω_cut_off Inf])
            y = 0
        end
        return y
    end
    if plot_bool
        ω = range(0, stop=ω_cut_off, length=1000)
        J_values = [J_drude(ω[i], 2) for i in 1:length(ω)]
        plot(ω, J_values, label="J(ω)", xlabel="ω (GHz)", ylabel="J(ω)")
    end
    return AB_drude, J_drude
end;

In [ ]:
function endsiteocc_d(t, cparams)
    es=cparams[1]
    ts=cparams[2]
    N=length(es)
    hmat = diagm(0=>es, 1=>ts, -1=>ts)
    F = eigen(hmat)
    U = F.vectors
    S = F.values
    real.(transpose(U[:,1].*exp.(im*t.*S))*U[:,N]*transpose(U[:,N])*(U[:,1].*exp.(-im*t.*S)))
end

function findchainlength_dich(T, cparams::Vector; eps=1e-4, verbose=false)
    Nmax = length(cparams[1])

    # Check if the full chain already fails the precision test
    occ_full = endsiteocc_d(T, [cparams[1][1:Nmax], cparams[2][1:Nmax-1]])
    occ_one_less = endsiteocc_d(T, [cparams[1][1:Nmax-1], cparams[2][1:Nmax-2]])
    if abs(occ_full - occ_one_less) > eps
        error("Suitable truncation not found, try larger starting chain length")
    end

    low = 1
    high = Nmax - 1
    best = Nmax - 1  # initialize to full chain minus 1

    while low <= high
        mid = (low + high) ÷ 2
        occ_mid = endsiteocc_d(T, [cparams[1][1:mid], cparams[2][1:mid-1]])
        occ_next = endsiteocc_d(T, [cparams[1][1:mid+1], cparams[2][1:mid]])

        verbose && println("Testing ntrunc = $mid")

        if abs(occ_mid - occ_next) > eps
            low = mid + 1  # too short: try longer
        else
            best = mid     # found acceptable length
            high = mid - 1 # try even shorter
        end
    end

    return best
end;

In [ ]:
function One_d_linear(n, bath, bath_params, t_max; ω_sys=5*2pi, κ=1/20, C_inv=0.147*2*pi, β=∞, energy=false)
    """
    INPUT :
    n : number of oscillators in the chain
    bath : function to define the spectral density J(ω) of the bath
    bath_params : parameters of the bath function
    ω_sys : system frequency in GHz (default: 5 * 2π)
    κ : decay rate of the bath in GHz (default: 1/20)
    C_inv : inverse of the capacitance in GHz (default: 0.147 * 2π)
    β : inverse temperature (default: ∞ for zero temperature T=0)
    energy : boolean to indicate if the energy should be calculated (default: false)

    OUTPUT :
    ϕ_sys : array of the system's ϕ_i values at each time step
    q_sys : array of the system's q_i values at each time step
    sol.t : time vector for the simulation
    E : array of the energy values at each time step if energy=true
    """

    AB, J = bath(bath_params=bath_params, κ=κ, C_inv=C_inv)
    onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

    ### Mapping ###
    Z_sys = C_inv / ω_sys # Characteristic impedance in GHz
    ω = vcat(ω_sys,onsite_energies) # GHz
    g_q=vcat(0, -hopping_energies) 
    g_ϕ=vcat(-system_chain_coupling*sqrt(Z_sys), hopping_energies) 

    ϕ_init = 1.0
    q_init = 1.0

    ϕ_0 = vcat(SVector(ϕ_init/sqrt(Z_sys)),SVector{n}(zeros(n)))
    q_0 = vcat(SVector(q_init*sqrt(Z_sys)),SVector{n}(zeros(n)))

    Mq=SymTridiagonal(ω, g_q)
    Mϕ=SymTridiagonal(ω, g_ϕ)

    A=Mq * Mϕ
    vals, vecs = eigen(A)

    ωk² = real(vals)
    ωk = sqrt.(ωk²)
    P = real(vecs)  # columns = normal modes

    P_inv=inv(P)

    # Project initial conditions into normal mode basis
    a0 = (P_inv * ϕ_0)
    adot0 = (P_inv * Mq * q_0)

    tspan = (0.0, t_max)
    dt=1/(20*maximum(ωk))
    npoints = Int(round((tspan[2]-tspan[1])/dt))  # number of time points


    # Time evolution in normal coordinates
    tlist = range(tspan[1], tspan[2], length=npoints)
    a_t = [a0 .* cos.(ωk .* t) .+ (adot0 ./ ωk) .* sin.(ωk .* t) for t in tlist]
    a_t_dot= [a0 .* (-ωk) .* sin.(ωk .* t) .+ (adot0) .* cos.(ωk .* t) for t in tlist]

    # Reconstruct real-space fields
    ϕ_t = [P * a for a in a_t]*sqrt(Z_sys)
    inv_Mq= inv(Mq)
    q_t = [inv_Mq * P * a for a in a_t_dot] / sqrt(Z_sys)  # q = \dot{ϕ}/Mq

    ϕ_sys =  [ϕ_t[i][1] for i in 1:length(tlist)]
    q_sys =  [q_t[i][1] for i in 1:length(tlist)];



    if energy
        E = 0.5 * [q_t[t]' * (Mq*Z_sys) * q_t[t] + ϕ_t[t]' * (Mϕ/Z_sys) * ϕ_t[t] for t in 1:npoints]
        return(ϕ_sys, q_sys, sol.t, E)
    end
    return(ϕ_sys, q_sys, tlist)
end

function CL_linear(t_max; ω_sys=5*2pi, κ=1/20, C_inv=0.147*2*pi)
    """
    INPUT :
    ω_sys : system frequency in GHz (default: 5 * 2π)
    κ : damping rate in GHz
    C_inv : inverse capacitance in GHz

    OUTPUT :
    ϕ_analytic : array of the time evolution of the phase ϕ(t)
    q_analytic : array of the time evolution of the charge q(t)
    t : time array
    """
    ϕ_init = 1.0
    q_init = 1.0
    X_0=[ϕ_init,q_init]
    
    tspan = (0.0, t_max)
    dt=1/(20*ω_sys)
    npoints = Int(round((tspan[2]-tspan[1])/dt))  # number of time points
    # Time evolution in normal coordinates
    t = range(tspan[1], tspan[2], length=npoints)

    Ω=sqrt(ω_sys^2-κ^2)
    q_analytic= @. exp(-κ*t)*(X_0[2]*cos(Ω*t) - (κ*X_0[2]+ω_sys^2*X_0[1]/C_inv)*sin(Ω*t)/Ω)
    ϕ_analytic= @. exp(-κ*t)*(X_0[1]*cos(Ω*t) + (X_0[2]*C_inv+κ*X_0[1])*sin(Ω*t)/Ω);

    return(ϕ_analytic, q_analytic, t)
end;

# Ohmic bath

In [ ]:
n_huge=700
AB, J = Ohmic_bath(bath_params=(ωc_dwn=0.0, ωc_up=10*2*pi),κ=1/20, C_inv=0.147*2*pi)
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=false)
n=n_opt+10;

In [ ]:
ϕ_1d, q_1d, t_1d = One_d_linear(n, Ohmic_bath, (ωc_dwn=0.0, ωc_up=10*2*pi), t_max);

In [ ]:
ϕ_CL, q_CL, t_CL = CL_linear(t_max)

In [ ]:
plot(ϕ_1d, q_1d, xlabel="ϕ", ylabel="q", label="")

In [ ]:
plot(t_1d, ϕ_1d, xlabel="t", ylabel="ϕ", label="")

In [ ]:
E_list, spectrum_list=comparison_simu([(ϕ_1d, q_1d, t_1d), (ϕ_CL, q_CL, t_CL)], Energy_linear, (L, C_inv))

E_1d = E_list[1]
E_CL = E_list[2]

(F_1d,freqs_1d) = spectrum_list[1]
(F_CL,freqs_CL) = spectrum_list[2]

plot(t_1d.*κ, E_1d, label="1d-chain")
plot!(t_CL.*κ, E_CL, label="CL master equation")

In [ ]:
plot(freqs_1d, abs.(F_1d), label = "1d-chain")
plot!(freqs_CL, abs.(F_CL), label = "CL master equation")
plot!(xlims=(3,8))

# Ohmic bath with exponential decay

In [ ]:
n_huge=1200
AB, J = Exp_decay(bath_params=(ω_c=50*2pi, ω_cut_off=100))
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=false)
n=n_opt+10;

In [ ]:
ϕ_1d_e, q_1d_e, t_1d_e = One_d_linear(n, Exp_decay, (ω_c=50*2pi, ω_cut_off=100), t_max);

In [ ]:
plot(ϕ_1d_e, q_1d_e, xlabel="ϕ", ylabel="q", label="")


In [ ]:
plot(t_1d_e*κ, ϕ_1d_e, label="Exponential decay")
plot!(t_CL*κ, ϕ_CL, label="CL master equation")
plot!(xlabel="κt", ylabel="ϕ")

# Drude-Lorentz bath

In [ ]:
n_huge=1100
AB, J = Drude_lorentz()
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength(t_max, params; eps=10^-4, verbose=false)
n=n_opt+10;

In [ ]:
ϕ_1d_l, q_1d_l, t_1d_l = One_d_linear(n, Drude_lorentz, (Ω=50*2pi, ω_cut_off=100), t_max);

In [ ]:
plot(ϕ_1d_l, q_1d_l, xlabel="ϕ", ylabel="q", label="")

# Comparison

In [ ]:
simu=(ϕ_1d, q_1d, t_1d)
simu_CL=(ϕ_CL, q_CL, t_CL)
simu_e=(ϕ_1d_e, q_1d_e, t_1d_e)
simu_DL=(ϕ_1d_l, q_1d_l, t_1d_l)
E_list, spectrum_list = comparison_simu([simu, simu_CL, simu_e, simu_DL], Energy_linear, (L, C_inv))
E = E_list[1]
E_CL = E_list[2]
E_e = E_list[3]
E_DL = E_list[4]
(F_1d,freqs_1d) = spectrum_list[1]
(F_CL,freqs_CL) = spectrum_list[2]
(F_e,freqs_e) = spectrum_list[3]
(F_DL,freqs_DL) = spectrum_list[4];

In [ ]:
theoretical_decay = exp.(-2κ*t_CL)E_CL[1]

In [ ]:
change_linewidth_thick()

In [ ]:
plot(t_1d.*κ, E, label="1d-chain")
plot!(t_CL.*κ, E_CL, label="CL")
plot!(t_1d_e.*κ, E_e, label="1d-chain exp decay")
plot!(t_1d_l.*κ, E_DL, label="1d-chain Drude-Lorentz")
plot!(t_CL.*κ, theoretical_decay, label=L"E(0)\exp(-2\kappa t)", linestyle=:dash)
plot!(xlabel="κt", ylabel="Energy", yaxis=:log10,legendfontsize=6)
#savefig("energy_comparison_CL_baths_linear.pdf")

In [ ]:
plot(freqs_1d, abs.(F_1d), label = "1d-chain")
plot!(freqs_CL, abs.(F_CL), label = "CL")
plot!(freqs_e, abs.(F_e), label = "1d-chain exp decay")
plot!(freqs_DL, abs.(F_DL), label = "1d-chain Drude-Lorentz")
plot!(xlims=(3,10.1), xlabel="Frequency (GHz)", ylabel="Amplitude")
vline!([2*pi], label="", color=:black, linestyle=:dash)
#savefig("frequency_comparison_CL_baths_linear.pdf")

In [ ]:
plot(freqs_e, abs.(F_e), label = "1d-chain exp decay")
plot!(freqs_DL, abs.(F_DL), label = "1d-chain Drude-Lorentz")
plot!(yaxis=:log, xlims=(8,11), xlabel="Frequency (GHz)", ylabel="Amplitude")
vline!([2.5pi], label="", color=:black, linestyle=:dash)